In [1]:
# Q4 — FIND EXACT PHASE-1 TRAINING FILE PATHS
from huggingface_hub import HfApi, get_token

HF_TOKEN = get_token()

print("Token available:", HF_TOKEN is not None)

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset"
)

print("\n" + "=" * 70)
print("PHASE 1 TRAINING FILES")
print("=" * 70)

for f in files:
    if "Phase_1_Pretraining/Train/" in f:
        print(f)

print("=" * 70)

Token available: False

PHASE 1 TRAINING FILES
Phase_1_Pretraining/Train/train_acts.json
Phase_1_Pretraining/Train/train_high_courts.json
Phase_1_Pretraining/Train/train_supreme_court.json
Phase_1_Pretraining/Train/train_tribunals.json


In [2]:
# ================================================================
# A4 — ENCODER-DECODER
# HIGH-SPEED PHASE-1 TRAINING
# ================================================================

import os
import json
import time
import math
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer
from torch.amp import autocast, GradScaler


# ================================================================
# CONFIG
# ================================================================

DEVICE = "cuda"

VOCAB_SIZE = 50_000

ENC_LEN = 1024
DEC_LEN = 1024

HIDDEN = 512
HEADS = 8

ENC_LAYERS = 6
DEC_LAYERS = 6

FFN_SIZE = 2048

TARGET_TOKENS = 200_000_000

LR = 1e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

PRINT_EVERY = 100
SAVE_EVERY = 5000

CHECKPOINT_DIR = (
    "/kaggle/working/a4_checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)


# ================================================================
# CUDA SPEED
# ================================================================

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("=" * 70)
print("🚀 A4 ENCODER-DECODER")
print("=" * 70)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1e9,
        2
    ),
    "GB"
)

print("FP16 : ON")
print("TF32 : ON")
print("=" * 70)


# ================================================================
# HF TOKEN
# ================================================================

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

HF_TOKEN = secrets.get_secret(
    "HF_TOKEN"
)

print(
    "HF token loaded:",
    HF_TOKEN is not None
)


# ================================================================
# TOKENIZER
# ================================================================

TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mohan567/tokenizer/"
    "tokenizer.json"
)

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

print(
    "✅ Tokenizer loaded"
)


# ================================================================
# A4 MODEL
# ================================================================

class A4Model(nn.Module):

    def __init__(self):

        super().__init__()


        # Shared embedding
        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN
        )


        # Separate positional embeddings
        self.encoder_position = nn.Embedding(
            ENC_LEN,
            HIDDEN
        )

        self.decoder_position = nn.Embedding(
            DEC_LEN,
            HIDDEN
        )


        # Encoder
        encoder_layer = nn.TransformerEncoderLayer(

            d_model=HIDDEN,

            nhead=HEADS,

            dim_feedforward=FFN_SIZE,

            dropout=0.1,

            activation="gelu",

            batch_first=True,

            norm_first=True

        )


        self.encoder = nn.TransformerEncoder(

            encoder_layer,

            num_layers=ENC_LAYERS

        )


        # Decoder
        decoder_layer = nn.TransformerDecoderLayer(

            d_model=HIDDEN,

            nhead=HEADS,

            dim_feedforward=FFN_SIZE,

            dropout=0.1,

            activation="gelu",

            batch_first=True,

            norm_first=True

        )


        self.decoder = nn.TransformerDecoder(

            decoder_layer,

            num_layers=DEC_LAYERS

        )


        self.encoder_norm = nn.LayerNorm(
            HIDDEN
        )

        self.decoder_norm = nn.LayerNorm(
            HIDDEN
        )


    def forward(
        self,
        encoder_ids,
        decoder_ids
    ):

        B, S = encoder_ids.shape

        _, T = decoder_ids.shape


        # --------------------------------------------------------
        # Encoder
        # --------------------------------------------------------

        enc_pos = torch.arange(
            S,
            device=encoder_ids.device
        )


        enc = (

            self.embedding(
                encoder_ids
            )

            +

            self.encoder_position(
                enc_pos
            )

        )


        enc = self.encoder(
            enc
        )


        enc = self.encoder_norm(
            enc
        )


        # --------------------------------------------------------
        # Decoder
        # --------------------------------------------------------

        dec_pos = torch.arange(
            T,
            device=decoder_ids.device
        )


        dec = (

            self.embedding(
                decoder_ids
            )

            +

            self.decoder_position(
                dec_pos
            )

        )


        # Causal mask
        causal_mask = torch.triu(

            torch.ones(
                T,
                T,
                device=decoder_ids.device,
                dtype=torch.bool
            ),

            diagonal=1

        )


        dec = self.decoder(

            dec,

            enc,

            tgt_mask=causal_mask

        )


        dec = self.decoder_norm(
            dec
        )


        # Shared output embedding
        logits = F.linear(

            dec,

            self.embedding.weight

        )


        return logits


# ================================================================
# CREATE MODEL
# ================================================================

model = A4Model().to(
    DEVICE
)


params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "\nA4 Parameters:",
    f"{params:,}"
)

print(
    "A4 Parameters:",
    f"{params / 1e6:.2f}M"
)


# ================================================================
# OPTIMIZER
# ================================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY,

    fused=True

)


scaler = GradScaler(
    "cuda"
)


# ================================================================
# DATA STREAM
# ================================================================

REPO_ID = (
    "Exploration-Lab/CS-781-Capstone"
)

TRAIN_FILES = [

    "Phase_1_Pretraining/Train/Acts.jsonl",

    "Phase_1_Pretraining/Train/High_Courts.jsonl",

    "Phase_1_Pretraining/Train/Supreme_Court.jsonl",

    "Phase_1_Pretraining/Train/Tribunals.jsonl"

]


def open_stream(filename):

    url = (

        "https://huggingface.co/datasets/"

        f"{REPO_ID}/resolve/main/"

        f"{filename}"

    )


    headers = {

        "Authorization":
        f"Bearer {HF_TOKEN}"

    }


    response = requests.get(

        url,

        headers=headers,

        stream=True,

        timeout=120

    )


    response.raise_for_status()

    return response


# ================================================================
# TOKEN STREAM
# ================================================================

def token_stream():

    while True:

        for filename in TRAIN_FILES:

            print(
                f"\n🌐 Opening {filename}"
            )

            response = open_stream(
                filename
            )


            for line in response.iter_lines(
                decode_unicode=True
            ):

                if not line:
                    continue


                try:

                    obj = json.loads(
                        line
                    )


                    text = (

                        obj.get("text")

                        or

                        obj.get("content")

                        or ""

                    )


                    if not text:
                        continue


                    ids = tokenizer.encode(
                        text
                    ).ids


                    for token in ids:

                        yield token


                except Exception:

                    continue


stream = token_stream()


# ================================================================
# GET TRAINING SAMPLE
# ================================================================

def get_sample():

    ids = []


    while len(ids) < 2049:

        ids.append(
            next(stream)
        )


    # 1024 encoder tokens
    encoder_ids = ids[
        :1024
    ]


    # decoder input
    decoder_ids = ids[
        1024:2048
    ]


    # decoder target
    targets = ids[
        1025:2049
    ]


    encoder_ids = torch.tensor(

        encoder_ids,

        dtype=torch.long,

        device=DEVICE

    ).unsqueeze(0)


    decoder_ids = torch.tensor(

        decoder_ids,

        dtype=torch.long,

        device=DEVICE

    ).unsqueeze(0)


    targets = torch.tensor(

        targets,

        dtype=torch.long,

        device=DEVICE

    ).unsqueeze(0)


    return (
        encoder_ids,
        decoder_ids,
        targets
    )


# ================================================================
# CHECKPOINT RESUME
# ================================================================

start_step = 0

files = [

    f

    for f in os.listdir(
        CHECKPOINT_DIR
    )

    if f.startswith("step_")

    and f.endswith(".pt")

]


if files:

    files.sort()

    latest = os.path.join(

        CHECKPOINT_DIR,

        files[-1]

    )


    print(
        "\n🔄 Loading:",
        latest
    )


    ckpt = torch.load(

        latest,

        map_location=DEVICE,

        weights_only=False

    )


    model.load_state_dict(
        ckpt["model"]
    )


    if "optimizer" in ckpt:

        optimizer.load_state_dict(
            ckpt["optimizer"]
        )


    if "scaler" in ckpt:

        scaler.load_state_dict(
            ckpt["scaler"]
        )


    start_step = ckpt[
        "step"
    ]


    print(
        "✅ Resumed from step:",
        start_step
    )

else:

    print(
        "\n🆕 Starting A4 from scratch"
    )


# ================================================================
# TRAINING
# ================================================================

TOKENS_PER_STEP = 1024

TOTAL_STEPS = math.ceil(

    TARGET_TOKENS
    /
    TOKENS_PER_STEP

)


print("\n" + "=" * 70)

print(
    "🚀 A4 TRAINING START"
)

print("=" * 70)

print(
    "Target tokens:",
    f"{TARGET_TOKENS:,}"
)

print(
    "Total steps:",
    f"{TOTAL_STEPS:,}"
)

print(
    "Starting step:",
    start_step
)

print("=" * 70)


model.train()

start_time = time.time()

last_time = start_time

last_tokens = (
    start_step * TOKENS_PER_STEP
)


for step in range(

    start_step + 1,

    TOTAL_STEPS + 1

):


    encoder_ids, decoder_ids, targets = (
        get_sample()
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    with autocast(

        device_type="cuda",

        dtype=torch.float16

    ):


        logits = model(

            encoder_ids,

            decoder_ids

        )


        loss = F.cross_entropy(

            logits.reshape(
                -1,
                VOCAB_SIZE
            ),

            targets.reshape(
                -1
            )

        )


    scaler.scale(
        loss
    ).backward()


    scaler.unscale_(
        optimizer
    )


    torch.nn.utils.clip_grad_norm_(

        model.parameters(),

        MAX_GRAD_NORM

    )


    scaler.step(
        optimizer
    )

    scaler.update()


    # ------------------------------------------------------------
    # SPEED
    # ------------------------------------------------------------

    if step % PRINT_EVERY == 0:

        now = time.time()

        elapsed = (
            now - last_time
        )

        current_tokens = (
            step * TOKENS_PER_STEP
        )

        new_tokens = (
            current_tokens
            -
            last_tokens
        )

        speed = (
            new_tokens
            /
            elapsed
        )

        remaining = (
            TARGET_TOKENS
            -
            current_tokens
        )

        eta = (
            remaining
            /
            speed
            /
            3600
            if speed > 0
            else 0
        )

        progress = (

            current_tokens
            /
            TARGET_TOKENS
            *
            100

        )


        print(

            f"Step {step:,}/"
            f"{TOTAL_STEPS:,} | "

            f"{progress:.2f}% | "

            f"Tokens "
            f"{current_tokens:,} | "

            f"Loss "
            f"{loss.item():.4f} | "

            f"{speed:,.0f} tok/s | "

            f"ETA "
            f"{eta:.2f} h"

        )


        last_time = now

        last_tokens = (
            current_tokens
        )


    # ------------------------------------------------------------
    # SAVE
    # ------------------------------------------------------------

    if step % SAVE_EVERY == 0:

        path = os.path.join(

            CHECKPOINT_DIR,

            f"step_{step:07d}.pt"

        )


        torch.save(

            {

                "step":
                step,

                "tokens":
                step * TOKENS_PER_STEP,

                "model":
                model.state_dict(),

                "optimizer":
                optimizer.state_dict(),

                "scaler":
                scaler.state_dict(),

                "architecture":
                "A4_encoder_decoder",

                "params":
                params

            },

            path

        )


        print(
            "💾 Saved:",
            path
        )


# ================================================================
# FINAL
# ================================================================

final_path = os.path.join(

    CHECKPOINT_DIR,

    "a4_final.pt"

)


torch.save(

    {

        "step":
        TOTAL_STEPS,

        "tokens":
        TARGET_TOKENS,

        "model":
        model.state_dict(),

        "architecture":
        "A4_encoder_decoder",

        "params":
        params

    },

    final_path

)


print("\n" + "=" * 70)

print(
    "🎉 A4 TRAINING COMPLETE"
)

print("=" * 70)

print(
    "Model:",
    final_path
)

print("=" * 70)

🚀 A4 ENCODER-DECODER


AssertionError: Torch not compiled with CUDA enabled

In [3]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

HF_TOKEN = secrets.get_secret("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True


In [5]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Train/train_acts.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

print("Acts file downloaded successfully")
print(path)

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Acts file downloaded successfully
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json


In [8]:
# ============================================================
# A4 — CORRECTED PRODUCTION TRAINING
# Persistent local data cache
# No repeated HTTP streaming
# ============================================================

import os
import json
import time
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download

# ============================================================
# CONFIG
# ============================================================

DEVICE = "cuda"

TOKEN_BUDGET = 200_000_000

ENC_LEN = 1024
DEC_LEN = 1024

TOKENS_PER_SAMPLE = ENC_LEN + DEC_LEN

BATCH_SIZE = 1
GRAD_ACCUM = 4

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

CHECKPOINT_EVERY = 5000

CACHE_DIR = "/kaggle/working/a4_data_cache"
CHECKPOINT_DIR = "/kaggle/working/a4_checkpoints"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

# ============================================================
# SPEED
# ============================================================

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ============================================================
# DOWNLOAD EACH FILE ONCE
# ============================================================

def download_source(filename):

    local_name = os.path.basename(filename)
    local_path = os.path.join(
        CACHE_DIR,
        local_name
    )

    if os.path.exists(local_path):
        return local_path

    print("Downloading:", local_name)

    path = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=filename,
        repo_type="dataset"
    )

    # Copy into persistent working directory
    import shutil

    shutil.copy2(
        path,
        local_path
    )

    return local_path


# ============================================================
# EXTRACT TEXT
# ============================================================

def extract_texts(obj):

    if isinstance(obj, str):
        return [obj]

    if isinstance(obj, list):

        result = []

        for item in obj:
            result.extend(
                extract_texts(item)
            )

        return result

    if isinstance(obj, dict):

        result = []

        # Prefer actual text fields
        for key in (
            "text",
            "content",
            "body"
        ):

            value = obj.get(key)

            if isinstance(value, str):
                result.append(value)

        if result:
            return result

        for value in obj.values():

            result.extend(
                extract_texts(value)
            )

        return result

    return []


# ============================================================
# TOKENIZE ONE SOURCE
# ============================================================

def tokenize_source(path):

    cache_file = path + ".tokens.pt"

    if os.path.exists(cache_file):

        return torch.load(
            cache_file,
            map_location="cpu",
            weights_only=True
        )

    print("Tokenizing:", os.path.basename(path))

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    texts = extract_texts(data)

    tokens = []

    for text in texts:

        ids = tokenizer.encode(
            text
        ).ids

        if len(ids) > 0:
            tokens.extend(ids)

    tokens = torch.tensor(
        tokens,
        dtype=torch.long
    )

    torch.save(
        tokens,
        cache_file
    )

    return tokens


# ============================================================
# PREPARE ALL FOUR SOURCES
# ============================================================

all_tokens = []

for filename in TRAIN_FILES:

    path = download_source(
        filename
    )

    tokens = tokenize_source(
        path
    )

    all_tokens.append(tokens)

# ============================================================
# COMBINE
# ============================================================

training_tokens = torch.cat(
    all_tokens,
    dim=0
)

del all_tokens

print(
    "Total available tokens:",
    len(training_tokens)
)

# ============================================================
# TRAINING PARAMETERS
# ============================================================

# Each sample:
#
# encoder = 1024 tokens
# decoder input = 1024 tokens
# decoder target = 1024 tokens
#
# We need 2049 tokens from a continuous stream.

SAMPLE_LEN = ENC_LEN + DEC_LEN + 1

steps = TOKEN_BUDGET // (
    BATCH_SIZE
    * GRAD_ACCUM
    * DEC_LEN
)

print("A4 parameters:",
      sum(p.numel() for p in model.parameters()))

print("A4 parameters (M):",
      sum(p.numel() for p in model.parameters()) / 1e6)

print("Target tokens:", TOKEN_BUDGET)

print("Total steps:", steps)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
    fused=True
)

# ============================================================
# AMP
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda"
)

# ============================================================
# SAMPLE FUNCTION
# ============================================================

position = 0

total_available = len(
    training_tokens
)


def get_sample():

    global position

    if position + SAMPLE_LEN >= total_available:

        position = 0

    chunk = training_tokens[
        position:
        position + SAMPLE_LEN
    ]

    position += DEC_LEN

    encoder_ids = chunk[
        :ENC_LEN
    ]

    decoder_ids = chunk[
        ENC_LEN:
        ENC_LEN + DEC_LEN
    ]

    targets = chunk[
        ENC_LEN + 1:
        ENC_LEN + DEC_LEN + 1
    ]

    return (
        encoder_ids,
        decoder_ids,
        targets
    )


# ============================================================
# TRAIN
# ============================================================

model.train()

optimizer.zero_grad(
    set_to_none=True
)

start_time = time.time()

tokens_seen = 0

for step in range(
    1,
    steps + 1
):

    for micro in range(
        GRAD_ACCUM
    ):

        encoder_batch = []
        decoder_batch = []
        target_batch = []

        for _ in range(
            BATCH_SIZE
        ):

            e, d, t = get_sample()

            encoder_batch.append(e)
            decoder_batch.append(d)
            target_batch.append(t)

        encoder_ids = torch.stack(
            encoder_batch
        ).to(
            DEVICE,
            non_blocking=True
        )

        decoder_ids = torch.stack(
            decoder_batch
        ).to(
            DEVICE,
            non_blocking=True
        )

        targets = torch.stack(
            target_batch
        ).to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(
                encoder_ids,
                decoder_ids
            )

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                targets.reshape(-1)
            )

            loss = loss / GRAD_ACCUM

        scaler.scale(
            loss
        ).backward()

        tokens_seen += (
            BATCH_SIZE * DEC_LEN
        )

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    optimizer.zero_grad(
        set_to_none=True
    )

    # ========================================================
    # CHECKPOINT + OCCASIONAL LOG
    # ========================================================

    if step % CHECKPOINT_EVERY == 0:

        elapsed = (
            time.time()
            - start_time
        )

        speed = (
            tokens_seen
            / elapsed
        )

        print(
            f"Step {step}/{steps} | "
            f"Tokens {tokens_seen:,} | "
            f"Speed {speed:.0f} tok/s | "
            f"Loss {loss.item() * GRAD_ACCUM:.4f}"
        )

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"step_{step:06d}.pt"
        )

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "step": step,
                "tokens_seen": tokens_seen,
                "config": {
                    "vocab_size": VOCAB_SIZE,
                    "encoder_len": ENC_LEN,
                    "decoder_len": DEC_LEN,
                    "hidden_size": 512,
                    "encoder_layers": 6,
                    "decoder_layers": 6,
                    "heads": 8,
                    "ffn_size": 2048
                }
            },
            checkpoint_path
        )


# ============================================================
# FINAL
# ============================================================

final_path = os.path.join(
    CHECKPOINT_DIR,
    "a4_final.pt"
)

torch.save(
    {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": steps,
        "tokens_seen": tokens_seen
    },
    final_path
)

elapsed = (
    time.time()
    - start_time
)

print()
print("========================================")
print("A4 TRAINING COMPLETE")
print("========================================")
print("Steps :", steps)
print("Tokens:", tokens_seen)
print("Time  :", round(elapsed / 3600, 2), "hours")
print("Model :", final_path)

Tokenizing: train_acts.json


NameError: name 'tokenizer' is not defined